# Тест на робастность: устойчивость к свободной речи

Перефразируем test через Mistral (смысл сохранён) и смотрим, как падает качество.
Гипотеза: tf-idf чувствителен к точным словам и проседает сильнее, семантический
трансформер устойчивее. Это и есть реальное преимущество трансформера для продукта
(пользователь пишет жалобы свободным языком).

Перефразированный test — оценочный набор, на нём НЕ обучаемся. **Runtime → GPU.**

In [ ]:
%cd /content
!rm -rf anamnesis && git clone -q https://github.com/vinograd131/anamnesis.git
%cd anamnesis
!pip install -q transformers accelerate peft
!pip uninstall -q -y torchao

In [ ]:
import getpass, os
os.environ["MISTRAL_API_KEY"] = getpass.getpass("MISTRAL_API_KEY: ")

## 1. Перефразируем test (создаёт `data/test_para_v1.jsonl`)

In [ ]:
from src.robustness import main as paraphrase_test
paraphrase_test()

## 2. Baseline (tf-idf) — оригинал vs перефраз

In [ ]:
from src.baseline import main as baseline_main
baseline_main("test")
baseline_main("test_para")

## 3. Трансформер (RuBioRoBERTa) — оригинал vs перефраз

In [ ]:
from src.transformer_ft import evaluate_saved
evaluate_saved("test")
evaluate_saved("test_para")

## 4. Сравнение устойчивости

Чем меньше падение (Δ), тем устойчивее модель к свободной речи.

In [ ]:
import json
m = json.load(open("reports/metrics.json"))
print(f"{'модель':24s} {'test':>8s} {'test_para':>10s} {'Δ macro-F1':>12s}")
for name in ["baseline", "rubioroberta_ft"]:
    t = m[name]["test"]["macro_f1"]
    p = m[name]["test_para"]["macro_f1"]
    print(f"{name:24s} {t:8.3f} {p:10.3f} {p - t:+12.3f}")

## Сохранить в git

In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
!git config user.email "karinkakarik@mail.ru"
!git config user.name "Git_Karina"
!git add data/test_para_v1.jsonl reports/
!git commit -q -m "robustness test results"
!git pull -q --rebase https://github.com/vinograd131/anamnesis.git main
!git push -q https://{token}@github.com/vinograd131/anamnesis.git HEAD:main